In [ ]:
import pandas as pd
from collections import Counter
import re

# 1️⃣ CSV 파일 불러오기
file_path = "gln_cafe_data.csv" ##csv파일 경로 불러오기

# 인코딩 자동 감지
for enc in ["utf-8-sig", "utf-8", "cp949"]:
    try:
        df = pd.read_csv(file_path, encoding=enc)
        break
    except Exception as e:
        print(f"{enc} 인코딩 실패: {e}")
else:
    raise ValueError("CSV 파일을 읽지 못했습니다. 인코딩을 확인하세요.")

# 2️⃣ content 열 확인
if "content" not in df.columns:
    raise ValueError(f"'content' 열이 없습니다. 현재 열 목록: {list(df.columns)}")

texts = df["content"].dropna().astype(str)

# 3️⃣ 텍스트 정제 함수
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z가-힣\s]", " ", text)  # 영어/한글만 남기기
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 4️⃣ 전체 텍스트 합치고 단어 분리
all_text = " ".join(clean_text(t) for t in texts)
tokens = all_text.split()

# 5️⃣ 불용어(Stopwords) 설정
stopwords = set("""
the is and in of a to for on with at by an be are this that it as from or was were has have but not we you i
he she they them their our your my me do does did so if then than too very more most can could would should
about into over under after before between because while during out up down off only also just any each both same 그냥 합니다 
에 및 으로 에서 하다 했다 있다 없다 수 것 등 을 를 은 는 이 가 에게 와 과 도 만 듯 듯이 그리고 그러나 또는 또한 해서 하고 입니다 인데요 있는 com 있어요 naver blog 있습니다 
""".split())    ###불용어 맘대로 수정 가능~~

filtered = [w for w in tokens if w not in stopwords and len(w) > 1]

# 6️⃣ 단어 빈도 계산
counter = Counter(filtered)
top_n = 50 ##최대 50개 정렬
top_keywords = counter.most_common(top_n)

# 7️⃣ 표(DataFrame) 형태로 출력
result_df = pd.DataFrame(top_keywords, columns=["Keyword", "Frequency"])
print("\n📊 최빈 키워드 Top", top_n)
print(result_df.to_string(index=False))

# 8️⃣ CSV로 저장
output_path = "top_keywords_content_from_cafe_csv.csv" #출력 이름
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ 결과 저장 완료: {output_path}")
